# Injury Intelligence - Classification

Author: Angela Lekivetz

This notebook performs classification of injury narratives based on OSHA report sourced from [OSHA Accident and Injury Data](https://www.kaggle.com/datasets/ruqaiyaship/osha-accident-and-injury-data-1517). The narratives have been cleaned and lemmatized using [spaCy](https://spacy.io/) in the `notebooks/1_eda.ipynb` notebook.

Two models will be trained: 
- A baseline model is trained using TF-IDF + Logistic Regression
- DistilBERT fine-tune

The baseline model provides a benchmark for comparison, where if DistilBERT doesn't meaningfully improve the performance, the added complexity isn't justified. All experiments will be tracked with MLflow for reproducibility and comparison.

In [ ]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import mlflow.pytorch
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, get_scheduler
from tqdm.auto import tqdm

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

## 1. Load the Dataset

The cleaned and lemmatized dataset from `1_eda.ipynb` is loaded, containing columns such as abstract text, event type and keywords, and the cleaned and lemmatized text.

 `lemma_text` is used as input for the TF-IDF baseline model, while `clean_text` is used for the DistilBERT model.

In [ ]:
df = pd.read_csv('../data/osha_filtered.csv')
df.head()

## 2. Prepare Data

This step prepares the data for training by applying a label encoder to the classifier column `Event type`.

The data is then split 80/20 into training and testing sets, using a constant seed for reproducibility.

In [ ]:
# Encoding
le = LabelEncoder()
df['label'] = le.fit_transform(df['Event type'])

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    df['lemma_text'],
    df['label'],
    test_size=0.2,
    random_state=SEED,
    stratify=df['label']
)

In [ ]:
print(X_train.shape, X_test.shape)
print(pd.Series(y_train).value_counts())
print(le.classes_)

## 3. Baseline Model (TF-IDF + Logistic Regression)

Our baseline model is a Term Frequency-Inverse Document Frequency (TF-IDF) + Logistic Regression model. TF-IDF converts text into numbers by measuring the relative importance of each word in the text, penalizing words that are common across the entire set. Logistic regression is then used to predict the probability of an a class occurring based on the TF-IDF features.

This baseline model is simple, fast, and interpretable. 

In [ ]:
mlflow.set_tracking_uri('file:../mlruns')
mlflow.set_experiment('injury-classification')

with mlflow.start_run(run_name='tfidf-logreg'):
    # Vectorize
    tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), sublinear_tf=True)
    X_train_vec = tfidf.fit_transform(X_train)
    X_test_vec = tfidf.transform(X_test)

    # Train
    clf = LogisticRegression(max_iter=500, C=1.0, random_state=SEED)
    clf.fit(X_train_vec, y_train)

    # Evaluate
    y_pred = clf.predict(X_test_vec)
    acc = accuracy_score(y_test, y_pred)

    print(f'Accuracy: {acc:.4f}')
    print(classification_report(y_test, y_pred, target_names=le.classes_))

    # Log
    mlflow.log_param('model', 'tfidf-logreg')
    mlflow.log_param('max_features', 20000)
    mlflow.log_param('ngram_range', '(1,2)')
    mlflow.log_metric('accuracy', acc)
    mlflow.sklearn.log_model(clf, 'model')

## 4. DistilBERT Fine-tuned Model

We then use DistilBERT, a smaller version of BERT (Bidirectional Encoder Representations from Transformers), to fine-tune the model on our dataset. BERT understands contextual language, and therefore we can adapt its general language understanding capabilities to our specific dataset.

In [ ]:
class InjuryDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_len)
        self.labels = labels
    
    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

train_dataset = InjuryDataset(X_train.tolist(), y_train.to_list(), tokenizer, MAX_LEN)
test_dataset = InjuryDataset(X_test.tolist(), y_test.tolist(), tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

num_labels = len(le.classes_)
model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
model.to(device)

with mlflow.start_run(run_name='distilbert-finetune'):
    model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
    model.to(device)

    optimizer = AdamW(model.parameters(), lr=LR)
    num_training_steps = EPOCHS * len(train_loader)
    scheduler = get_scheduler('linear', optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

    mlflow.log_params({'model': MODEL_NAME, 'epochs': EPOCHS, 'lr': LR, 'batch_size': BATCH_SIZE, 'max_len': MAX_LEN})

    # Training loop
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}'):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        mlflow.log_metric('train_loss', avg_loss, step=epoch)
        print(f'Epoch {epoch+1} — Loss: {avg_loss:.4f}')

    # Evaluation
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc='Evaluating'):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            preds = torch.argmax(outputs.logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch['labels'].cpu().numpy())

    bert_acc = accuracy_score(all_labels, all_preds)
    print(f'\nDistilBERT Accuracy: {bert_acc:.4f}')
    print(classification_report(all_labels, all_preds, target_names=le.classes_))
    mlflow.log_metric('accuracy', bert_acc)

    # Save model
    model.save_pretrained('../data/distilbert_injury')
    tokenizer.save_pretrained('../data/distilbert_injury')
    print('Model saved.')

# 5. Compare Models

We see a modest improvement in accuracy when using the DistilBERT model (`79.35%`) vs the TF-IDF model (`78.22%`). This is expected on short, structured text. This gap isn't as pronounced as TF-IDF works well on short, consistent text with distinctive vocabulary such as "roof", "ladder", or "voltage". 

In [ ]:
print(f'TF-IDF + LogReg accuracy : {acc:.4f}')
print(f'DistilBERT accuracy      : {bert_acc:.4f}')
print(f'Improvement              : +{(bert_acc - acc)*100:.2f}pp')

## 6. Inference on Sample Narratives

To validate the model qualitatively, we test it on a few hand-crafted narratives with known expected classes. 

The model performs well on classes with distinctive vocabulary such as "caught in" and "fell from elevation", but struggles when narratives contain language associated with multiple event types.

In [ ]:
sample_texts = [
    ("the employee was operating a mechanical press when their hand was caught in the point of operation", "Caught in or between"),
    ("an employee was delivering materials when a load of lumber fell from the truck and struck them in the head", "Struck-by"),
    ("an employee was working on a roof when they fell approximately 15 feet to the ground below", "Fall (from elevation)")
]

model.eval()
inputs = tokenizer([t[0] for t in sample_texts], truncation=True, padding=True, max_length=MAX_LEN, return_tensors='pt')
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)
    preds = torch.argmax(outputs.logits, dim=-1)

for (text, expected), pred in zip(sample_texts, preds):
    print(f'Text:      {text[:80]}')
    print(f'Expected:  {expected}')
    print(f'Predicted: {le.classes_[pred]}')
    print()